In [5]:
import pandas as pd
import numpy as np

#carregando o dataset
file_inmet = '/content/drive/MyDrive/Pivic/dados/Dados/Institudo de Metereologia/dados 2023/INMET_NE_PB_A313_CAMPINA GRANDE_01-01-2023_A_31-12-2023.CSV'

df_clima = pd.read_csv(file_inmet,
                       sep=';',
                       decimal=',',
                       skiprows=8,
                       encoding='latin1')
# selecionando as colunas que serão úteis
colunas_map = {
    'Data': 'DATA',
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'CHUVA_TOTAL',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)': 'TEMP_MEDIA',
    'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'TEMP_MAX',
    'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'TEMP_MIN',
    'UMIDADE RELATIVA DO AR, HORARIA (%)': 'UMIDADE'
}
df_clima = df_clima.rename(columns=colunas_map)[colunas_map.values()]


#tratando irregularidade

df_clima['DATA'] = pd.to_datetime(df_clima['DATA'], format='%Y/%m/%d')


#transformando o valor em datetime para semana, pq coloquei os valores na tabela do dataset da epidemiologia em semanas
df_clima['SEMANA'] = df_clima['DATA'].dt.isocalendar().week


#salvando os parâmetros por semana
df_clima_semanal = df_clima.groupby('SEMANA').agg({
    'CHUVA_TOTAL': 'sum',
    'TEMP_MEDIA': 'mean',
    'TEMP_MAX': 'max',
    'TEMP_MIN': 'min',
    'UMIDADE': 'mean'
}).reset_index()

df_clima_semanal = df_clima_semanal.round(2)



df_dengue = pd.read_csv('/content/drive/MyDrive/Pivic/tratamento dados/dados tratados/epidemiologia/dengue2023_tratado.csv')

#transformando os valores em inteiro para que o cruzamento das bases não dê bronca
df_dengue['SEMANA EPDM'] = pd.to_numeric(df_dengue['SEMANA EPDM'], errors='coerce')

#criando uma coluna para agregar os casos em quantidade por semana
df_casos_semanal = df_dengue.groupby('SEMANA EPDM').size().reset_index(name='TOTAL_CASOS')
df_casos_semanal = df_casos_semanal.rename(columns={'SEMANA EPDM': 'SEMANA'})



#fazendo o merge propriamente dito, utilizando o inner, serve pra utilizar apenas colunas que estão presente nas duas tabelas
df_final = pd.merge(df_clima_semanal, df_casos_semanal, on='SEMANA', how='inner')

#salvandotratamento_clima2025.
#df_final.to_csv('dataset_clima/semana_epdm.csv', index=False)

df_final.head(10)

,SEMANA,CHUVA_TOTAL,TEMP_MEDIA,TEMP_MAX,TEMP_MIN,UMIDADE,TOTAL_CASOS
0,1,28.4,24.06,31.5,20.4,82.96,1
1,5,0.0,25.24,32.4,21.4,76.62,2
2,9,0.0,25.26,32.0,21.7,78.19,1
3,15,11.2,23.94,29.8,19.8,87.30,1
4,17,15.2,24.14,30.6,21.0,89.39,1
5,19,10.6,24.73,31.5,20.9,83.61,1
6,25,29.0,21.86,28.1,18.3,93.55,1
7,28,5.4,22.64,28.5,18.5,87.88,1
8,29,5.8,21.98,28.0,18.2,84.59,1
9,34,1.0,23.06,29.7,19.5,86.82,1


In [4]:
df_final.to_csv('dataset_clima_semana_epdm2023.csv', index=False)
df_clima_semanal.to_csv('dataset_clima_2023_tratato.csv',index=False)